# Computational Analysis: Monte Carlo, Bloom Filter & MCMC
# Statistical Audit — pandas-dev/pandas

### Research Question yang Dijawab:
- **RQ3:** Berapa probabilitas sebuah issue di pandas-dev/pandas membutuhkan lebih dari 30 hari untuk ditutup, diestimasi tanpa formula analitik?
- **Tambahan:** Bagaimana Bloom Filter dapat digunakan untuk deteksi bug report secara efisien? Kombinasi bug issues mana yang optimal diselesaikan dalam kapasitas waktu terbatas?

**Member:** Muhamad Bintang Ramadhan— Computational Analyst (Member E)

---

## AI Usage Disclosure

**Member:** Muhamad Bintang Ramadhan— Computational Analyst | **Tools used:** Claude

| Task | Tool | Prompt summary | Output modified? |
| :--- | :--- | :--- | :--- |
| Scaffolding struktur class BloomFilter | Claude | "Meminta kerangka implementasi Bloom Filter dengan k hash functions menggunakan hashlib" | Ya — logika `_get_positions`, pemilihan parameter k dan m, serta konteks dataset disesuaikan secara mandiri |
| Boilerplate loop simulasi Monte Carlo | Claude | "Meminta kerangka loop n_trials untuk estimasi probabilitas empiris" | Ya — `event_fn` berbasis `np.random.choice` dari distribusi empiris dan threshold 30 hari dirancang sendiri |
| Scaffolding struktur fungsi MCMC | Claude | "Meminta kerangka Metropolis-Hastings untuk knapsack problem" | Ya — definisi items dari data bug issues nyata, nilai capacity, dan acceptance criterion disesuaikan sendiri |

**Written entirely without AI:** Seluruh teks interpretasi hasil simulasi, justifikasi pemilihan teknik komputasi, narasi keterkaitan antar layer, dan paragraf ringkasan pada summary cell.

---
## Setup: Import Library dan Load Data

In [ ]:
import sys
sys.path.append("../")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.simulation import estimate_probability, make_issue_duration_event_fn, BloomFilter, mcmc_knapsack

# Load dataset bersih dari Member A
df = pd.read_csv('../data/clean/dataset.csv')

# Subset yang digunakan layer ini
issues        = df[df['type'] == 'issue'].copy()
closed_issues = issues['duration_days'].dropna()
bug_issues    = issues[(issues['is_bug'] == 1) & (issues['duration_days'].notna()) & (issues['duration_days'] > 0)]

print(f"Total issues    : {len(issues):,}")
print(f"Issues closed   : {len(closed_issues):,}")
print(f"Bug issues      : {len(issues[issues['is_bug']==1]):,}")
print(f"Bug issues (closed, >0 hari): {len(bug_issues):,}")

Dataset yang digunakan adalah output bersih dari Member A (Tsabita). Terdapat **3.001 issues** di dataset, dengan **1.929 issues** yang sudah ditutup dan memiliki data `duration_days`. Subset ini menjadi dasar seluruh simulasi pada notebook ini.

---
## Bagian 1 — Monte Carlo Simulation (RQ3)

### Justifikasi Pemilihan Teknik
RQ3 menanyakan probabilitas sebuah issue membutuhkan lebih dari 30 hari untuk ditutup. Pendekatan analitik (formula tertutup) membutuhkan asumsi distribusi yang kuat — misalnya distribusi Eksponensial atau Gamma — yang belum tentu sesuai dengan pola data nyata `pandas-dev/pandas`. Data `duration_days` pada dataset ini bersifat sangat skewed (mean 777 hari, median 297 hari), sehingga asumsi distribusi parametrik berisiko menghasilkan estimasi yang bias.

Monte Carlo merupakan alat yang tepat karena memungkinkan estimasi probabilitas langsung dari distribusi empiris data, tanpa perlu mengasumsikan bentuk distribusi tertentu. Dengan `n_trials = 50.000`, estimasi yang dihasilkan memiliki standar error yang sangat kecil dan konvergen secara stabil.

In [ ]:
# Buat event_fn berbasis distribusi empiris duration_days
event_fn = make_issue_duration_event_fn(
    duration_array=closed_issues.values,
    threshold=30,
    seed=42
)

# Jalankan simulasi Monte Carlo (50.000 trial)
mc_result = estimate_probability(event_fn, n_trials=50000, seed=42)

print("=== HASIL MONTE CARLO SIMULATION ===")
print(f"Estimasi P(duration > 30 hari) : {mc_result['probability']:.6f}")
print(f"Jumlah trial                   : {mc_result['n_trials']:,}")
print(f"Jumlah kejadian True (sukses)  : {mc_result['n_success']:,}")
print(f"Standard Error estimasi        : {mc_result['std_error']:.6f}")
print()
print(f"Interval estimasi (95%):")
p    = mc_result['probability']
se   = mc_result['std_error']
low  = p - 1.96 * se
high = p + 1.96 * se
print(f"  [{low:.6f}, {high:.6f}]")

In [ ]:
# Visualisasi konvergensi estimasi Monte Carlo
np.random.seed(42)
durations = np.array(closed_issues.values)
checkpoints = [100, 500, 1000, 5000, 10000, 20000, 50000]
running_probs = []

samples = np.random.choice(durations, size=50000, replace=True)
cumulative = np.cumsum(samples > 30)
trial_range = np.arange(1, 50001)
running_estimate = cumulative / trial_range

plt.figure(figsize=(10, 5))
plt.plot(trial_range, running_estimate, color='steelblue', linewidth=1, label='Estimasi berjalan')
plt.axhline(y=mc_result['probability'], color='red', linestyle='--', linewidth=1.5,
            label=f"Konvergen ke: {mc_result['probability']:.4f}")
plt.axhline(y=(durations > 30).mean(), color='green', linestyle=':', linewidth=1.5,
            label=f"P empiris sesungguhnya: {(durations>30).mean():.4f}")
plt.xlabel('Jumlah Trial')
plt.ylabel('Estimasi P(duration > 30 hari)')
plt.title('Konvergensi Estimasi Monte Carlo — P(Issue > 30 Hari)')
plt.legend()
plt.tight_layout()
plt.savefig('../report/fig_monte_carlo_convergence.png', dpi=150)
plt.show()

### Interpretasi Hasil Monte Carlo

Simulasi Monte Carlo dengan **50.000 trial** menghasilkan estimasi probabilitas sebesar **~0.6672** (66,72%). Artinya, sekitar **2 dari 3 issues** di repositori `pandas-dev/pandas` membutuhkan lebih dari 30 hari untuk ditutup sejak pertama kali dilaporkan.

Standard error estimasi sebesar **~0.0021** menunjukkan bahwa hasil simulasi ini sangat stabil — interval estimasi 95%-nya sempit di kisaran [0.6631, 0.6713]. Grafik konvergensi di atas memperlihatkan bahwa estimasi berjalan dengan cepat stabil setelah sekitar 5.000 trial, membuktikan bahwa pilihan `n_trials = 50.000` lebih dari cukup untuk menghasilkan estimasi yang andal.

Nilai ini konsisten dengan hasil eksplorasi Member A pada `01_eda.ipynb`, di mana median `duration_days` pada issues tercatat sebesar **297 hari** — jauh melampaui threshold 30 hari. Ini memperkuat temuan bahwa mayoritas issues di pandas memiliki siklus hidup yang panjang, kemungkinan karena kompleksitas teknis permasalahan yang dilaporkan.

---
## Bagian 2 — Bloom Filter

### Justifikasi Pemilihan Teknik
Dalam konteks audit repositori berskala besar seperti `pandas-dev/pandas`, terdapat kebutuhan praktis untuk mengecek apakah sebuah issue number tertentu pernah dilabeli sebagai **bug report** — misalnya saat memproses stream data baru dari GitHub API. Pendekatan naif (menyimpan seluruh set issue number di memori) tidak efisien untuk data berskala ribuan hingga jutaan entri.

Bloom Filter adalah alat yang tepat untuk kasus ini: struktur data probabilistik yang mampu menjawab pertanyaan keanggotaan (*membership query*) dengan penggunaan memori sangat kecil. Tradeoff-nya adalah kemungkinan **false positive** (item dinyatakan ada padahal tidak), namun **tidak pernah false negative** (jika filter menjawab "tidak ada", maka pasti tidak ada). Untuk use case audit di mana false negative lebih berbahaya dari false positive, Bloom Filter sangat sesuai.

Parameter yang digunakan:
- `m = 10.000` bit (ukuran bit array)
- `k = 5` fungsi hash
- Jumlah item yang dimasukkan: seluruh issue number berlabel bug

In [ ]:
# Inisialisasi Bloom Filter
bf = BloomFilter(k=5, m=10000)

# Masukkan semua issue number yang berlabel bug
bug_numbers = issues[issues['is_bug'] == 1]['number'].tolist()
for num in bug_numbers:
    bf.add(num)

n_inserted = len(bug_numbers)
theoretical_fpr = bf.theoretical_fpr(n_inserted)

print("=== BLOOM FILTER — SETUP ===")
print(f"Ukuran bit array (m)      : {bf.m:,}")
print(f"Jumlah hash functions (k) : {bf.k}")
print(f"Jumlah item dimasukkan (n): {n_inserted:,}")
print(f"FPR Teoritis              : {theoretical_fpr:.6f} ({theoretical_fpr*100:.4f}%)")
print(f"  Formula: (1 - (1 - 1/m)^n)^k  [Tsun 2020, p. 329]")

In [ ]:
# Uji membership: issue yang MEMANG bug (should return True)
test_bugs     = bug_numbers[:5]
# Uji membership: issue yang BUKAN bug (menguji false positive)
non_bug_numbers = issues[issues['is_bug'] == 0]['number'].tolist()
test_nonbugs  = non_bug_numbers[:10]

print("=== UJI MEMBERSHIP: BUG ISSUES (Expected: True) ===")
for num in test_bugs:
    result = bf.contains(num)
    print(f"  Issue #{num} → {result}")

print()
print("=== UJI MEMBERSHIP: NON-BUG ISSUES (Expected: False, mungkin FP) ===")
fp_count = 0
for num in test_nonbugs:
    result = bf.contains(num)
    status = "⚠ FALSE POSITIVE" if result else "OK"
    print(f"  Issue #{num} → {result}  {status}")
    if result:
        fp_count += 1

print()
print(f"False positive ditemukan: {fp_count} dari {len(test_nonbugs)} sampel non-bug")
print(f"FPR observasi sampel    : {fp_count/len(test_nonbugs):.4f}")
print(f"FPR teoritis            : {theoretical_fpr:.4f}")

In [ ]:
# Visualisasi FPR teoritis vs jumlah item yang dimasukkan
n_range = range(100, 3000, 50)
fpr_values = [bf.theoretical_fpr(n) for n in n_range]

plt.figure(figsize=(10, 5))
plt.plot(list(n_range), fpr_values, color='darkorange', linewidth=2)
plt.axvline(x=n_inserted, color='red', linestyle='--',
            label=f'n aktual = {n_inserted} (FPR = {theoretical_fpr:.4f})')
plt.xlabel('Jumlah Item Dimasukkan (n)')
plt.ylabel('False Positive Rate (FPR) Teoritis')
plt.title('Bloom Filter FPR vs Jumlah Item (m=10.000, k=5)')
plt.legend()
plt.tight_layout()
plt.savefig('../report/fig_bloom_filter_fpr.png', dpi=150)
plt.show()

### Interpretasi Hasil Bloom Filter

Bloom Filter berhasil diinisialisasi dengan `m = 10.000` bit dan `k = 5` fungsi hash, kemudian diisi dengan **1.369 issue number** berlabel bug dari dataset `pandas-dev/pandas`.

**FPR Teoritis** dihitung menggunakan formula dari Tsun (2020, p. 329):
$$\text{FPR} = \left(1 - \left(1 - \frac{1}{m}\right)^n\right)^k$$
Dengan parameter di atas, FPR teoritis yang diperoleh adalah **~0.0027 (0.27%)** — sangat rendah. Artinya, dari 10.000 query terhadap issue yang bukan bug, rata-rata hanya sekitar 27 yang akan salah diklasifikasikan sebagai bug.

Pengujian membership pada sampel bug issues menunjukkan semua item yang dimasukkan berhasil dideteksi (tidak ada false negative), sesuai dengan sifat dasar Bloom Filter. Grafik menunjukkan bahwa FPR meningkat secara non-linear seiring bertambahnya n — hal ini menginformasikan bahwa jika dataset issues tumbuh signifikan di masa depan, parameter `m` perlu ditingkatkan agar FPR tetap terjaga di bawah 1%.

---
## Bagian 3 — MCMC Knapsack

### Justifikasi Pemilihan Teknik
Tim maintainer `pandas-dev/pandas` menghadapi masalah alokasi sumber daya yang nyata: dengan kapasitas waktu terbatas, bug issues mana yang sebaiknya diprioritaskan untuk diselesaikan agar jumlah bug yang tertutup maksimal?

Ini adalah formulasi klasik **Knapsack Problem** — setiap bug issue memiliki *weight* (jumlah hari yang dibutuhkan untuk menyelesaikannya, berdasarkan `duration_days` historis) dan *value* (setiap bug yang selesai bernilai 1). Knapsack Problem bersifat NP-Hard, sehingga solusi eksak tidak praktis untuk data skala besar.

**MCMC dengan algoritma Metropolis-Hastings** adalah alat yang tepat karena memungkinkan eksplorasi ruang solusi yang sangat besar secara probabilistik — menemukan solusi mendekati optimal tanpa harus mengevaluasi semua kombinasi yang mungkin. Ini merupakan aplikasi langsung dari Week 14 (Tsun 2020, p. 331).

Skenario yang digunakan: 15 bug issues representatif dari dataset dengan variasi `duration_days`, kapasitas tim sebesar **9.000 hari** (setara ~25 tahun-orang, representatif untuk backlog jangka panjang repositori aktif).

In [ ]:
# Definisi items: 15 bug issues representatif dari dataset
# weight = duration_days historis, value = 1 (satu bug resolved)
items = [
    {'name': 'issue #63889', 'weight': 14,   'value': 1},
    {'name': 'issue #63167', 'weight': 116,  'value': 1},
    {'name': 'issue #62520', 'weight': 5,    'value': 1},
    {'name': 'issue #61917', 'weight': 127,  'value': 1},
    {'name': 'issue #61222', 'weight': 4,    'value': 1},
    {'name': 'issue #60589', 'weight': 308,  'value': 1},
    {'name': 'issue #58495', 'weight': 441,  'value': 1},
    {'name': 'issue #55345', 'weight': 942,  'value': 1},
    {'name': 'issue #51044', 'weight': 1160, 'value': 1},
    {'name': 'issue #46331', 'weight': 1474, 'value': 1},
    {'name': 'issue #41563', 'weight': 1779, 'value': 1},
    {'name': 'issue #35342', 'weight': 2079, 'value': 1},
    {'name': 'issue #26853', 'weight': 2342, 'value': 1},
    {'name': 'issue #22497', 'weight': 2780, 'value': 1},
    {'name': 'issue #16756', 'weight': 2962, 'value': 1},
]

capacity = 9000  # Total kapasitas hari tim maintainer

print("=== ITEMS — BUG ISSUES REPRESENTATIF ===")
print(f"{'Nama':<18} {'Weight (hari)':>14} {'Value':>7}")
print("-" * 42)
for item in items:
    print(f"{item['name']:<18} {item['weight']:>14} {item['value']:>7}")
print("-" * 42)
print(f"{'Total weight'::<18} {sum(i['weight'] for i in items):>14}")
print(f"Kapasitas (capacity)  : {capacity:,} hari")

In [ ]:
# Jalankan MCMC Knapsack
mcmc_result = mcmc_knapsack(
    items=items,
    capacity=capacity,
    n_iter=100000,
    seed=42
)

print("=== HASIL MCMC KNAPSACK ===")
print(f"Jumlah iterasi MCMC   : {mcmc_result['n_iter']:,}")
print(f"Acceptance rate       : {mcmc_result['acceptance_rate']:.4f} ({mcmc_result['acceptance_rate']*100:.2f}%)")
print(f"Total value optimal   : {mcmc_result['best_value']} bug issues")
print(f"Total weight dipakai  : {mcmc_result['best_weight']:,} hari dari {capacity:,} hari")
print()
print("Bug issues dalam solusi optimal:")
for name in mcmc_result['best_items']:
    w = next(i['weight'] for i in items if i['name'] == name)
    print(f"  {name} ({w} hari)")

print()
print("Bug issues TIDAK dipilih (di luar kapasitas):")
not_selected = [i['name'] for i in items if i['name'] not in mcmc_result['best_items']]
for name in not_selected:
    w = next(i['weight'] for i in items if i['name'] == name)
    print(f"  {name} ({w} hari)")

In [ ]:
# Visualisasi: weight comparison selected vs not selected
selected     = mcmc_result['best_items']
colors       = ['steelblue' if i['name'] in selected else 'lightcoral' for i in items]
names        = [i['name'].replace('issue ', '') for i in items]
weights      = [i['weight'] for i in items]

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.barh(names, weights, color=colors)
ax.axvline(x=capacity, color='red', linestyle='--', linewidth=1.5,
           label=f'Kapasitas total = {capacity:,} hari')
ax.set_xlabel('Duration (hari)')
ax.set_title('MCMC Knapsack — Seleksi Bug Issues Optimal\n(Biru = dipilih, Merah = tidak dipilih)')

from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='steelblue', label='Dipilih MCMC'),
    Patch(facecolor='lightcoral', label='Tidak dipilih'),
]
ax.legend(handles=legend_elements)
plt.tight_layout()
plt.savefig('../report/fig_mcmc_knapsack.png', dpi=150)
plt.show()

### Interpretasi Hasil MCMC Knapsack

MCMC dengan **100.000 iterasi** berhasil mengidentifikasi kombinasi bug issues yang memaksimalkan jumlah bug terselesaikan dalam kapasitas **9.000 hari**. Algoritma Metropolis-Hastings mengeksplorasi ruang solusi secara probabilistik: setiap iterasi memproposalkan perubahan (flip satu item), menerima perubahan yang meningkatkan nilai, dan sesekali menerima perubahan yang lebih buruk untuk menghindari jebakan solusi lokal.

**Acceptance rate** yang diperoleh mencerminkan efisiensi eksplorasi MCMC — proporsi proposal yang diterima selama proses berjalan. Nilai ini menunjukkan bahwa algoritma berhasil menyeimbangkan antara eksplorasi ruang solusi baru dan eksploitasi solusi yang sudah baik.

Secara praktis, hasil MCMC ini memberikan rekomendasi berbasis data kepada maintainer `pandas-dev/pandas`: alih-alih mencoba menyelesaikan semua bug secara acak, tim dapat memprioritaskan kombinasi issues yang secara historis membutuhkan waktu lebih pendek, sehingga jumlah bug yang berhasil ditutup dalam kapasitas tertentu menjadi maksimal.

Perlu dicatat bahwa `duration_days` historis digunakan sebagai proxy untuk estimasi effort — asumsi ini valid sebagai baseline, meskipun effort aktual dapat dipengaruhi faktor lain seperti kompleksitas kode dan ketersediaan kontributor.

---
## Summary — Keterkaitan dengan Layer Sebelumnya dan Temuan Keseluruhan

Layer komputasi ini merupakan layer terakhir dalam pipeline audit statistik `pandas-dev/pandas`. Ketiga teknik yang digunakan masing-masing menjawab dimensi berbeda dari kesehatan repositori:

**1. Monte Carlo (RQ3):** Mengkonfirmasi estimasi dari Member B dan Member C bahwa isu waktu penyelesaian di pandas sangat signifikan. Probabilitas **~66.72%** issue membutuhkan lebih dari 30 hari adalah temuan kunci audit ini. Ini melengkapi temuan Member D (RQ1) bahwa meskipun merge rate PR tinggi (62.09%), backlog issues memiliki siklus hidup yang sangat panjang — keduanya bersama-sama menggambarkan repositori yang aktif menerima kontribusi namun kewalahan dalam mengelola laporan bug yang masuk.

**2. Bloom Filter:** Membuktikan bahwa sistem deteksi bug report berbasis probabilistic data structure dapat diimplementasikan dengan FPR teoritis serendah **~0.27%** menggunakan Bloom Filter berukuran 10.000 bit. Ini relevan sebagai rekomendasi tooling untuk tim maintainer dalam menskalakan proses triage.

**3. MCMC Knapsack:** Memberikan perspektif optimasi yang tidak bisa dijawab oleh layer estimasi maupun inferensia: bukan hanya *berapa* probabilitasnya, tapi *apa* yang sebaiknya dilakukan dengan sumber daya terbatas. Hasil MCMC menghasilkan rekomendasi konkret berbasis data untuk prioritisasi backlog.

Seluruh temuan layer ini akan diintegrasikan ke dalam **Section: Computational Analysis** pada laporan akhir kelompok.